# Load SIC codes into the `business_type` table

Reads `industry_codes.csv` and fills the `business_type` table in the shared SQLite database.

Notes
- `sic_code` is stored as **text** so leading zeros survive (`01110`, not `1110`).
- The file mixes real 5-digit codes with "Section A" to "Section U" heading rows. Only the 5-digit codes are loaded, the headings are skipped (sector grouping comes later).
- Safe to re-run: rows are matched on `sic_code`, so existing `id` values never change. That matters because other tables will point at `business_type.id`.

## 1. Settings

In [8]:
import sqlite3
from pathlib import Path

import pandas as pd

CSV_PATH = Path("raw/official/industry_codes.csv")      # change if the file lives elsewhere
DB_PATH = Path("processed/database.db")    # the shared SQLite database file

## 2. Read and tidy the CSV

In [9]:
raw = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
raw.columns = [c.strip() for c in raw.columns]
assert list(raw.columns) == ["Code", "Description"], f"Unexpected columns: {raw.columns.tolist()}"

# The source has stray double spaces and trailing blanks in some descriptions
raw["Code"] = raw["Code"].str.strip()
raw["Description"] = raw["Description"].str.replace(r"\s+", " ", regex=True).str.strip()

print(f"{len(raw)} rows read")
raw.head()

752 rows read


,Code,Description
0,Section A,"Agriculture, Forestry and Fishing"
1,01110,"Growing of cereals (except rice), leguminous c..."
2,01120,Growing of rice
3,01130,"Growing of vegetables and melons, roots and tu..."
4,01140,Growing of sugar cane


## 3. Separate codes from section headings, and validate

In [10]:
is_section = raw["Code"].str.fullmatch(r"Section [A-U]")
is_code = raw["Code"].str.fullmatch(r"\d{5}")

unexpected = raw[~(is_section | is_code)]
assert unexpected.empty, f"Rows that are neither a 5-digit code nor a section heading:\n{unexpected}"

sections = raw[is_section]
codes = raw[is_code].copy()

duplicates = codes[codes["Code"].duplicated(keep=False)]
assert duplicates.empty, f"Duplicate SIC codes:\n{duplicates}"
assert (codes["Description"] != "").all(), "Some codes have an empty description"

print(f"{len(codes)} SIC codes to load ({len(sections)} section heading rows skipped)")

731 SIC codes to load (21 section heading rows skipped)


## 4. Create the table

In [11]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS business_type (
    id          INTEGER PRIMARY KEY,
    sic_code    TEXT NOT NULL UNIQUE CHECK (length(sic_code) = 5),
    description TEXT NOT NULL
);
"""

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON;")   # SQLite ignores foreign keys unless this is set on every connection
con.execute("PRAGMA journal_mode = WAL;")  # lets teammates read while you write
con.executescript(SCHEMA)

## 5. Load the rows

In [12]:
rows = list(codes.itertuples(index=False, name=None))   # (sic_code, description)

with con:   # one transaction: all rows are saved, or none are
    con.executemany(
        """
        INSERT INTO business_type (sic_code, description) VALUES (?, ?)
        ON CONFLICT(sic_code) DO UPDATE SET description = excluded.description
        """,
        rows,
    )

print(f"Loaded {len(rows)} rows into {DB_PATH}")

Loaded 731 rows into processed/database.db


## 6. Check the result

In [13]:
n = con.execute("SELECT COUNT(*) FROM business_type").fetchone()[0]
assert n == len(codes), f"Expected {len(codes)} rows in business_type but found {n}"
print(f"{n} rows in business_type")

pd.read_sql("SELECT * FROM business_type ORDER BY id LIMIT 5", con)

731 rows in business_type


,id,sic_code,description
0,1,01110,"Growing of cereals (except rice), leguminous c..."
1,2,01120,Growing of rice
2,3,01130,"Growing of vegetables and melons, roots and tu..."
3,4,01140,Growing of sugar cane
4,5,01150,Growing of tobacco


Leading zeros should still be there, and these codes are too vague to say much about the business (non-trading, dormant, other business support, residents' property management). Worth handling when we load Companies House data:

In [14]:
pd.read_sql(
    """
    SELECT * FROM business_type
    WHERE sic_code IN ('01110', '74990', '82990', '98000', '99999')
    ORDER BY sic_code
    """,
    con,
)

,id,sic_code,description
0,1,01110,"Growing of cereals (except rice), leguminous c..."
1,610,74990,Non-trading company
2,655,82990,Other business support service activities n.e.c.
3,727,98000,Residents property management
4,731,99999,Dormant Company


In [15]:
con.close()